In [1]:
pip install faker pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.3 MB/s eta 0:00:00


In [2]:
import random
from datetime import timedelta
from faker import Faker
import pandas as pd

In [3]:
fake = Faker()



In [4]:
# Seed for reproducibility
random.seed(42)
Faker.seed(42)

In [5]:
# Configuration
NUM_ROWS = 5000  # Your target row count for the ML models
suppliers = ["Tokyo Logistics Co", "Kyoto Tech Components", "Osaka Freight Corp", "Nagoya Auto Parts"]
locations = ["Tokyo Port", "Yokohama Port", "Osaka Port", "Nagoya Airport", "Bengaluru Hub", "Chennai Port"]
carriers = ["Sea Freight", "Air Cargo", "Express Road"]
statuses = ["Delivered", "Delayed", "In-Transit"]

data_list = []

In [6]:
for _ in range(NUM_ROWS):
    # 1. Generate core dates
    dispatch_date = fake.date_time_between(start_date="-1y", end_date="now")

    # Transit days logic (randomized between 5 to 20 days)
    expected_transit_days = random.randint(5, 15)
    expected_date = dispatch_date + timedelta(days=expected_transit_days)

    # Simulate delays for historical data
    # Let's say 20% of shipments get delayed
    is_delayed = random.random() < 0.20

    if is_delayed:
        # Actual date is longer than expected date
        actual_transit_days = expected_transit_days + random.randint(3, 10)
        status = "Delayed"
    else:
        # Arrived on time or early
        actual_transit_days = expected_transit_days - random.randint(0, 2)
        status = "Delivered"

    actual_date = dispatch_date + timedelta(days=actual_transit_days)

    # For active/in-transit shipments (simulate current view)
    if random.random() < 0.05:
        status = "In-Transit"
        actual_date = None  # Not delivered yet

    # 2. Build the row dictionary
    row = {
        "product_id": f"PROD-{fake.unique.random_number(digits=6)}",
        "supplier_name": random.choice(suppliers),
        "origin_location": random.choice(locations[:4]),  # Japanese ports
        "destination": random.choice(locations[4:]),  # Indian hubs
        "quantity": random.randint(100, 5000),
        "dispatch_date": dispatch_date.strftime("%Y-%m-%d %H:%M:%S"),
        "expected_date": expected_date.strftime("%Y-%m-%d %H:%M:%S"),
        "actual_date": actual_date.strftime("%Y-%m-%d %H:%M:%S")
        if actual_date
        else "NaT",
        "carrier_type": random.choice(carriers),
        "status": status,
    }
    data_list.append(row)

In [8]:
# 3. Convert to Pandas DataFrame and save to CSV
df = pd.DataFrame(data_list)
df.to_csv("blockchain_supply_chain_data.csv", index=False)

print(f"Success! Generated {NUM_ROWS} rows of supply chain data.")
print(df.head())

Success! Generated 5000 rows of supply chain data.
    product_id          supplier_name origin_location    destination  \
0   PROD-26225  Kyoto Tech Components      Tokyo Port  Bengaluru Hub   
1  PROD-256787     Tokyo Logistics Co   Yokohama Port   Chennai Port   
2  PROD-772246  Kyoto Tech Components  Nagoya Airport   Chennai Port   
3  PROD-776646      Nagoya Auto Parts      Tokyo Port   Chennai Port   
4   PROD-91161     Tokyo Logistics Co  Nagoya Airport  Bengaluru Hub   

   quantity        dispatch_date        expected_date          actual_date  \
0      4937  2026-02-14 18:21:24  2026-03-01 18:21:24  2026-03-08 18:21:24   
1      1905  2026-03-24 01:32:55  2026-03-29 01:32:55  2026-04-04 01:32:55   
2      2376  2025-09-15 17:54:24  2025-09-29 17:54:24  2025-09-29 17:54:24   
3      2917  2025-08-02 15:44:35  2025-08-10 15:44:35  2025-08-09 15:44:35   
4      4622  2026-05-18 01:55:37  2026-05-27 01:55:37  2026-05-25 01:55:37   

   carrier_type     status  
0     Air Cargo   